In [5]:
import tensorflow as tf
import math, os, random, cv2, numpy, torch
import torch.nn as nn
import pytest
import torch
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
from ai_edge_litert.interpreter import Interpreter  # LiteRT
import inspect

In [5]:
# Check insides of .tflite
path = "/Users/helbuk/Library/Mobile Documents/com~apple~CloudDocs/NTNU/NTNU_Thesis_2026/Compiler-Aware_Model_Optimization/Thesis_Compiler_Aware_Model_Optimization_2026/thesis2026-project/models/yolov8n_saved_model/yolov8n_float32.tflite"

interpreter = Interpreter(model_path=path)
interpreter.allocate_tensors()

print("Inputs:")
for d in interpreter.get_input_details():
    print(d["name"], d["shape"], d["dtype"])

print("\nOutputs:")
for d in interpreter.get_output_details():
    print(d["name"], d["shape"], d["dtype"])

ops = interpreter._get_ops_details()
print("\n#ops =", len(ops))
for i, op in enumerate(ops[:40]):
    print(i, op["op_name"])

Inputs:
images [  1 640 640   3] <class 'numpy.float32'>

Outputs:
Identity [   1   84 8400] <class 'numpy.float32'>

#ops = 257
0 PAD
1 CONV_2D
2 LOGISTIC
3 MUL
4 PAD
5 CONV_2D
6 LOGISTIC
7 MUL
8 CONV_2D
9 LOGISTIC
10 MUL
11 STRIDED_SLICE
12 STRIDED_SLICE
13 CONV_2D
14 LOGISTIC
15 MUL
16 CONV_2D
17 LOGISTIC
18 MUL
19 ADD
20 CONCATENATION
21 CONV_2D
22 LOGISTIC
23 MUL
24 PAD
25 CONV_2D
26 LOGISTIC
27 MUL
28 CONV_2D
29 LOGISTIC
30 MUL
31 STRIDED_SLICE
32 STRIDED_SLICE
33 CONV_2D
34 LOGISTIC
35 MUL
36 CONV_2D
37 LOGISTIC
38 MUL
39 ADD


In [ ]:
# Check insides of .tflite
path = "/Users/helbuk/Library/Mobile Documents/com~apple~CloudDocs/NTNU/NTNU_Thesis_2026/Compiler-Aware_Model_Optimization/Thesis_Compiler_Aware_Model_Optimization_2026/thesis2026-project/models/yolov8n_saved_model/yolov8n_float32.tflite"

interpreter = Interpreter(model_path=path)
interpreter.allocate_tensors()

print("Inputs:")
for d in interpreter.get_input_details():
    print(d["name"], d["shape"], d["dtype"])

print("\nOutputs:")
for d in interpreter.get_output_details():
    print(d["name"], d["shape"], d["dtype"])

ops = interpreter._get_ops_details()
print("\n#ops =", len(ops))
for i, op in enumerate(ops[:40]):
    print(i, op["op_name"])

In [2]:
from ultralytics import YOLO

PATH_to_model = '/Users/helbuk/Library/Mobile Documents/com~apple~CloudDocs/NTNU/NTNU_Thesis_2026/Compiler-Aware_Model_Optimization/Thesis_Compiler_Aware_Model_Optimization_2026/thesis2026-project/models/yolov8n.pt'
PATH_to_data = '/Users/helbuk/Library/Mobile Documents/com~apple~CloudDocs/NTNU/NTNU_Thesis_2026/Compiler-Aware_Model_Optimization/Thesis_Compiler_Aware_Model_Optimization_2026/thesis2026-project/datasets/coco/data.yaml'
SAVE_TO = '/Users/helbuk/Library/Mobile Documents/com~apple~CloudDocs/NTNU/NTNU_Thesis_2026/Compiler-Aware_Model_Optimization/Thesis_Compiler_Aware_Model_Optimization_2026/thesis2026-project/models/yolov8n_models/tflite/'
IMG_EXAMPLE = '/Users/helbuk/Library/Mobile Documents/com~apple~CloudDocs/NTNU/NTNU_Thesis_2026/Compiler-Aware_Model_Optimization/Thesis_Compiler_Aware_Model_Optimization_2026/thesis2026-project/datasets/coco/valid/images/000000000139.jpg'
# Load the YOLO26 model
model = YOLO(PATH_to_model)

In [3]:
help(model.export)

Help on method export in module ultralytics.engine.model:

export(**kwargs: 'Any') -> 'str' method of ultralytics.models.yolo.model.YOLO instance
    Export the model to a different format suitable for deployment.
    
    This method facilitates the export of the model to various formats (e.g., ONNX, TorchScript) for deployment
    purposes. It uses the 'Exporter' class for the export process, combining model-specific overrides, method
    defaults, and any additional arguments provided.
    
    Args:
        **kwargs (Any): Arbitrary keyword arguments for export configuration. Common options include:
            - format (str): Export format (e.g., 'onnx', 'engine', 'coreml').
            - half (bool): Export model in half-precision.
            - int8 (bool): Export model in int8 precision.
            - device (str): Device to run the export on.
            - workspace (int): Maximum memory workspace size for TensorRT engines.
            - nms (bool): Add Non-Maximum Suppression

In [ ]:
# Export the model to TFLite format
model.export(format="tflite", imgsz=640, int8=True, nms=False, data=PATH_to_data, device='cpu', split="train", fraction=0.004, simplify=True, opset=12, verbose=False)  # creates 'yolo26n_float32.tflite'

# Load the exported TFLite model
tflite_model = YOLO(SAVE_TO + "yolov8n_int8_20_percent_calib_640imgsz.tflite")

# Run inference
results = tflite_model(IMG_EXAMPLE)
results

Ultralytics 8.4.6 🚀 Python-3.11.14 torch-2.9.1 CPU (Apple M3 Pro)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from '/Users/helbuk/Library/Mobile Documents/com~apple~CloudDocs/NTNU/NTNU_Thesis_2026/Compiler-Aware_Model_Optimization/Thesis_Compiler_Aware_Model_Optimization_2026/thesis2026-project/models/yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)

TensorFlow SavedModel: starting export with tensorflow 2.20.0...
TensorFlow SavedModel: collecting INT8 calibration images from 'data=/Users/helbuk/Library/Mobile Documents/com~apple~CloudDocs/NTNU/NTNU_Thesis_2026/Compiler-Aware_Model_Optimization/Thesis_Compiler_Aware_Model_Optimization_2026/thesis2026-project/datasets/coco/data.yaml'
Fast image access ✅ (ping: 0.1±0.1 ms, read: 351.7±265.0 MB/s, size: 124.5 KB)
Scanning /Users/helbuk/Library/Mobile Documents/com~apple~CloudDocs/NTNU/NTNU_Thesis_2026/Compiler-Aware_Model_Optimization/The

I0000 00:00:1771429824.150931 12971936 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1771429824.151586 12971936 single_machine.cc:376] Starting new session
W0000 00:00:1771429824.603263 12971936 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1771429824.603275 12971936 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1771429824.892557 12971936 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1771429824.892622 12971936 single_machine.cc:376] Starting new session
W0000 00:00:1771429825.472781 12971936 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1771429825.472791 12971936 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1771429825.701770 12971936 devices.cc:76] 

W0000 00:00:1771429828.850320 12971936 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1771429828.850334 12971936 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1771429828.862694 12971936 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32
W0000 00:00:1771430297.231801 12971936 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1771430297.231817 12971936 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
W0000 00:00:1771430762.510116 12971936 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1771430762.510129 12971936 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
